In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

import pandas as pd

df = pd.read_pickle('/content/drive/MyDrive/EE/df_clean.pkl')
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size = 0.2, random_state = 42, shuffle = True)

# Model 1


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd


class ChessFinalBoardDatasetWithELO(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        game_tensor = self.df.iloc[idx]['game_tensor']
        label = int(self.df.iloc[idx]['result'])
        final_board = game_tensor[-1]

        white_elo = self.df.iloc[idx]['white_elo']
        black_elo = self.df.iloc[idx]['black_elo']
        elo_diff = torch.tensor([(white_elo - black_elo) / 1000.0], dtype = torch.float32)

        return final_board.float(), elo_diff, torch.tensor(label, dtype = torch.long)

In [ ]:
class ChessCNNWithELO(nn.Module):
    def __init__(self):
        super(ChessCNNWithELO, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(12, 32, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.Flatten()
        )
        self.mlp = nn.Sequential(
            nn.Linear(64 * 8 * 8 + 1, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, board, elo_diff):
        features = self.cnn(board)
        combined = torch.cat([features, elo_diff], dim = 1)
        return self.mlp(combined)



dataset_train = ChessFinalBoardDatasetWithELO(train_df)
train_loader = DataLoader(dataset_train, batch_size = 8, shuffle=True, num_workers = 0)
dataset_test = ChessFinalBoardDatasetWithELO(test_df)
test_loader = DataLoader(dataset_test, batch_size = 8, shuffle=False, num_workers = 0)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ChessCNNWithELO().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 1e-4)

In [ ]:
# Train
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for boards, elo_diffs, labels in train_loader:
        boards = boards.to(device)
        elo_diffs = elo_diffs.to(device)
        labels = labels.to(device)

        outputs = model(boards, elo_diffs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    total = 0
    correct = 0
    # Evaluation
    model.eval()
    with torch.no_grad():
      for boards, elo_diffs, labels in test_loader:
          boards = boards.to(device)
          elo_diffs = elo_diffs.to(device)
          labels = labels.to(device)
          outputs = model(boards, elo_diffs)
          _, predicted = torch.max(outputs, dim=1)
          correct += (predicted == labels).sum().item()
          total += labels.size(0)
    test_acc = 100 * correct / total
    if(test_acc >= 64.05):
      torch.save(model, f"model_at_{test_acc:.2f}.pt")
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss:.4f}, Accuracy: {test_acc:.2f}%")

Epoch [1/20], Loss: 6330.0297, Accuracy: 64.15%
Epoch [2/20], Loss: 6221.2235, Accuracy: 63.98%
Epoch [3/20], Loss: 6204.5999, Accuracy: 63.70%
Epoch [4/20], Loss: 6183.2268, Accuracy: 63.94%
Epoch [5/20], Loss: 6149.9941, Accuracy: 63.88%
Epoch [6/20], Loss: 6109.3796, Accuracy: 63.67%
Epoch [7/20], Loss: 6059.4053, Accuracy: 63.38%
Epoch [8/20], Loss: 5990.9374, Accuracy: 63.06%
Epoch [9/20], Loss: 5906.2558, Accuracy: 62.77%
Epoch [10/20], Loss: 5803.9520, Accuracy: 62.02%
Epoch [11/20], Loss: 5674.6842, Accuracy: 62.11%
Epoch [12/20], Loss: 5494.2113, Accuracy: 60.35%
Epoch [13/20], Loss: 5280.7199, Accuracy: 61.45%
Epoch [14/20], Loss: 5033.0480, Accuracy: 60.45%
Epoch [15/20], Loss: 4751.9514, Accuracy: 60.50%
Epoch [16/20], Loss: 4447.2020, Accuracy: 59.25%
Epoch [17/20], Loss: 4125.4504, Accuracy: 60.12%
Epoch [18/20], Loss: 3796.8917, Accuracy: 59.21%


KeyboardInterrupt: 

# Model 2

In [ ]:
import torch
import torch.nn as nn

class ChessCNNRNNModel(nn.Module):
    def __init__(self):
        super(ChessCNNRNNModel, self).__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(12, 32, kernel_size = 3, padding = 1),
            nn.GELU(),
            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.GELU(),
            nn.Flatten()
        )

        self.cnn_output_size = 64 * 8 * 8
        self.rnn_input_size = self.cnn_output_size + 1

        self.rnn = nn.GRU(input_size = self.rnn_input_size, hidden_size = 256, batch_first = True)

        self.mlp = nn.Sequential(
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Linear(128, 2)
        )

    def forward(self, board_tensor, evals):
        """
        board_tensor: (B, T, 12, 8, 8)
        evals: (B, T, 1)
        """
        B, T, C, H, W = board_tensor.shape
        boards_reshaped = board_tensor.view(B * T, C, H, W)
        cnn_features = self.cnn(boards_reshaped).view(B, T, -1)
        combined_features = torch.cat([cnn_features, evals], dim = -1)
        rnn_out, _ = self.rnn(combined_features)
        final_feature = rnn_out[:, -1, :]
        return self.mlp(final_feature)


In [ ]:
from torch.utils.data import Dataset
import torch

class ChessGameDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        game_tensor = self.df.iloc[idx]['game_tensor']
        eval_tensor = self.df.iloc[idx]['eval_tensor']
        label = int(self.df.iloc[idx]['result'])


        if eval_tensor.ndim == 1:
            eval_tensor = eval_tensor.unsqueeze(-1)

        return game_tensor.float(), eval_tensor.float(), torch.tensor(label, dtype = torch.long)


In [ ]:
chess_dataset = ChessGameDataset(train_df)
validation = ChessGameDataset(test_df)


torch.Size([17, 12, 8, 8])
torch.Size([17, 1])
tensor(1)


In [ ]:
from torch.nn.utils.rnn import pad_sequence

def pad_collate(batch):
    """
    Pads game_tensor and eval_tensor along the time dimension.
    """
    game_tensors, eval_tensors, labels = zip(*batch)


    padded_game = pad_sequence(game_tensors, batch_first = True)
    padded_eval = pad_sequence(eval_tensors, batch_first = True)

    labels = torch.stack(labels)
    return padded_game, padded_eval, labels


In [ ]:
from torch.utils.data import DataLoader
train_loader = DataLoader(chess_dataset, batch_size = 1024, shuffle = True, collate_fn = pad_collate)
test_loader = DataLoader(validation, batch_size = 1024, shuffle = True, collate_fn = pad_collate)

train

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ChessCNNRNNModel().to(device)
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 8e-4, weight_decay = 1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, verbose=True
)
all_eval_tensors = []

for _, eval_tensor, _ in train_loader:
    all_eval_tensors.append(eval_tensor)

all_eval_tensors = torch.cat(all_eval_tensors, dim = 0)

eval_mean = all_eval_tensors.mean()
eval_std = all_eval_tensors.std()
eval_mean = eval_mean.to(device)
eval_std = eval_std.to(device)

all_eval_tensors = []

for _, eval_tensor, _ in test_loader:
    all_eval_tensors.append(eval_tensor)

all_eval_tensors = torch.cat(all_eval_tensors, dim = 0)

eval_mean_test = all_eval_tensors.mean()
eval_std_test = all_eval_tensors.std()
eval_mean_test = eval_mean.to(device)
eval_std_test = eval_std.to(device)


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [ ]:


num_epochs = 40
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for board_tensor, eval_tensor, labels in train_loader:
        board_tensor = board_tensor.to(device)
        eval_tensor = eval_tensor.to(device)
        eval_tensor = (eval_tensor - eval_mean) / eval_std
        labels = labels.to(device)

        outputs = model(board_tensor, eval_tensor)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_accuracy = 100 * correct / total

    # Evaluation
    model.eval()
    test_correct = 0
    with torch.no_grad():
        for board_tensor, eval_tensor, labels in test_loader:
            board_tensor = board_tensor.to(device)
            eval_tensor = eval_tensor.to(device)
            eval_tensor = (eval_tensor - eval_mean_test) / eval_std_test
            labels = labels.to(device)
            outputs = model(board_tensor, eval_tensor)
            _, predicted = torch.max(outputs, dim=1)
            test_correct += (predicted == labels).sum().item()

    test_accuracy = 100 * test_correct / len(test_loader.dataset)
    scheduler.step(test_accuracy)
    if(test_accuracy >= 57.9):
      torch.save(model, f"model_at_{test_accuracy:.2f}.pt")
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss:.4f}, "
          f"Train Accuracy: {train_accuracy:.2f}%, Test Accuracy: {test_accuracy:.2f}%")


Epoch [1/40], Loss: 51.6507, Train Accuracy: 58.46%, Test Accuracy: 57.87%
Epoch [2/40], Loss: 51.5085, Train Accuracy: 58.50%, Test Accuracy: 58.20%
Epoch [3/40], Loss: 51.4586, Train Accuracy: 58.60%, Test Accuracy: 57.58%
Epoch [4/40], Loss: 51.3997, Train Accuracy: 58.84%, Test Accuracy: 57.19%
Epoch [5/40], Loss: 51.3193, Train Accuracy: 59.01%, Test Accuracy: 57.64%
Epoch [6/40], Loss: 51.2093, Train Accuracy: 59.26%, Test Accuracy: 57.40%
Epoch [7/40], Loss: 50.8244, Train Accuracy: 60.03%, Test Accuracy: 57.70%
Epoch [8/40], Loss: 50.6599, Train Accuracy: 60.21%, Test Accuracy: 57.48%


KeyboardInterrupt: 